# Differentiable Artificial Metabolic Networks


This notebook demonstrates the pipeline for differentiable Artificial Metabolic Networks.

Using a large-scale dataset of *Escherichia coli* growth under well-defined laboratory conditions (Aida & Ying, 2025). The purpose if to use this dataset for training an Artificial Metabolic Network that can learn to map `media concentration` $\rightarrow$ `growth` and `uptake flux`

---

Aida, H., & Ying, B. (2025). Population Dynamics of Escherichia coli Growing under Chemically Defined Media. Scientific Data, 12(1), 984. https://doi.org/10.1038/s41597-025-05356-3

## Set up environment

In [1]:
# Automatic module reloading
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import r2_score

# Set up paths
from pathlib import Path

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
DATA_EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
MODEL_DIR = PROJECT_ROOT / 'models'


In [2]:
import torch

# Suppress Python-level warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPUs available: {torch.cuda.device_count()}")

PyTorch Version: 2.0.1+cu117


## Step 1. Aida & Ying Dataset Pre-processing
#### 1.1. Match the Aida & Ying Compounds to iML1515 reactions

In [3]:
from src.data_preprocess import *

In [4]:
# iML1515 model
model_path = DATA_RAW_DIR / "iML1515.xml"
model = cobra.io.read_sbml_model(model_path)

In [5]:
# Map Aida & Ying compounds to iML1515 reactions
model_path = DATA_RAW_DIR / "iML1515.xml"
mapping_csv = create_supplementary_csv(model, DATA_INTERIM_DIR)

Total AIDA compounds in mapping: 40


In [6]:
# Create media dataframe
mapping_csv = DATA_INTERIM_DIR / "aidaying_compound_mapping.csv"
final_df = create_media_growth_csv(DATA_EXTERNAL_DIR, mapping_csv, DATA_INTERIM_DIR)

Mapped 40 AIDA columns into 37 GEM reactions.
6 media columns were not mapped.
Columns not renamed: ['Label', 'Assay ID', 'FolicAcid (mM)', 'Riboflavin (mM)', 'H3BO3 (mM)', 'AminobenzoicAcid (mM)']
Media compositions before dropping duplicates → 13608
Unique media compositions after dropping duplicates → 941
Total growth entries in CSV: 13608
Entries with r_info = 1: 262
Entries with r_info = 2: 475
Growth entries remaining (r_info = 0): 12871
Final unique conditions → 924
Final media data saved to /scratch/chinasse/BactoCat-AMN/data/interim/aidaying_media_growth_amns.csv


In [7]:
# Correct the csv to keep only the duplicated model media reactions
corrected_df = match_amns_media(DATA_INTERIM_DIR, DATA_INTERIM_DIR, final_df)

Columns matched and renamed with AMNS reference: 37/37
Final media data saved to /scratch/chinasse/BactoCat-AMN/data/interim/aidaying_media_growth_amns.csv


### 2. Create modeling dataset

In [8]:
from src.dataset import load_experimental_data, extract_model_matrices, duplicate_reversible_reactions, normalize_features

In [9]:
# Load experimental CSV
exp_data = load_experimental_data(
    csv_path=DATA_INTERIM_DIR / 'aidaying_media_growth_amns.csv',
    growth_column='GR_AVG')

In [10]:
# Extract model matrices from COBRA model
metabolic_model = extract_model_matrices(model, exp_data.medium_ids)

# Duplicate reversible reactions
irreversible_model = duplicate_reversible_reactions(model)

# Normalize features
normalized_data, params = normalize_features(exp_data, method='max')

The default model had 2712 reactions and the duplicated-reactions model has 3682 reactions.
